In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import pandas as pd
import torch
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain import PromptTemplate, LLMChain
from langchain_huggingface import HuggingFacePipeline


torch.cuda.empty_cache()

In [2]:
from pydantic import BaseModel, Field, conint, confloat
from enum import Enum
from typing import Optional

class SexoJuiz(str, Enum):
    MASCULINO = "Masculino"
    FEMININO = "Feminino"

class SimNao(str, Enum):
    SIM = "Sim"
    NAO = "Não"

class RegimeInicial(str, Enum):
    FECHADO = "Fechado"
    SEMIABERTO = "Semi-aberto"
    ABERTO = "Aberto"
    NONE = "None"

class SentencaModel(BaseModel):
    processo: str = Field(..., pattern=r'^\d{20}$')
    juiz: str
    sexo_juiz: SexoJuiz
    vara: str
    nome: str
    local: str
    maconha: Optional[str] = "None"
    maconha_g: Optional[confloat(ge=0)] = 0
    cocaina: Optional[str] = "None"
    cocaina_g: Optional[confloat(ge=0)] = 0
    crack: Optional[str] = "None"
    crack_g: Optional[confloat(ge=0)] = 0
    ecstasy: Optional[str] = "None"
    ecstasy_g: Optional[confloat(ge=0)] = 0
    lsd: Optional[str] = "None"
    lsd_g: Optional[confloat(ge=0)] = 0
    outras: Optional[str] = "None"
    anabolizantes: SimNao
    anorexigenos: SimNao
    haxixe: SimNao
    skank: SimNao
    lanca_perfume: SimNao
    tolueno: SimNao
    den_drog: str
    den_outros: Optional[str] = "None"
    sentenca: str
    res_drogas: str
    res_outros: Optional[str] = "None"
    pena_base: str
    agravantes33_agrup: SimNao
    confissao: SimNao
    menoridade: SimNao
    atenuantes33_agrup: SimNao
    adolescente: SimNao
    arma_de_fogo: SimNao
    interestadual: SimNao
    concurso_formal: SimNao
    estabelecimento: SimNao
    aumento33_agrup: SimNao
    paragrafo_4o_agrupado: str
    pena33: str
    pena33_meses: conint(ge=0)
    pena_drogas: str
    pena_outros: Optional[str] = "None"
    tot_pen: str
    tot_pen_meses: conint(ge=0)
    substituicao_da_pena: Optional[str] = "None"
    regime_inicial: RegimeInicial
    # Flags (todos obrigatórios com default False)
    flag_local_de_trafico: bool = False
    flag_preso_no_momento_da_sentenca: bool = False
    flag_confissao_informal: bool = False
    # ... (repetir para todas as flags com default=False)

In [3]:
from outlines import models, generate

PROMPT_TEMPLATE = """\
[INST] <<SYS>>
Você é um especialista jurídico. Extraia dados estruturados deste documento seguindo rigorosamente o schema.

**Regras Críticas:**
1. Campos numéricos: Sempre em gramas (apenas números)
2. Campos Sim/Não: Apenas "Sim" ou "Não"
3. Processo: Exatamente 20 dígitos
4. Flags: True apenas se explicitamente mencionado
5. Nada de markdown ou texto extra

<</SYS>>

## Documento:
{document}

## Schema Explicativo:
{format_instructions}

[/INST]

Resposta JSON: 
"""

In [ ]:
from outlines import models, generate
from pydantic import ValidationError

def criar_extrator():
    model = models.transformers("meta-llama/Llama-3.2-3B-Instruct")
    return generate.json(model, SentencaModel)

def parse_resposta(texto: str, max_retries=3):
    extrator = criar_extrator()
    for _ in range(max_retries):
        try:
            resposta = extrator(PROMPT_TEMPLATE.format(
                document=texto,
                format_instructions=SentencaModel.schema_json(indent=2)
            ))
            return resposta.model_dump()
        except ValidationError as e:
            print(f"Erro na tentativa {_+1}: {e}")
    raise ValueError("Falha após 3 tentativas")


In [6]:
df_test = pd.read_parquet("validation.parquet")[0:4]

In [5]:
resultado = parse_resposta(documento_exemplo)
print(resultado)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_2881086/3739652588.py:14: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  format_instructions=SentencaModel.schema_json(indent=2)


{'processo': '00316684320178260050', 'juiz': 'Maria Silva', 'sexo_juiz': <SexoJuiz.FEMININO: 'Feminino'>, 'vara': '12', 'nome': 'Joao da Silva', 'local': 'Favela do Jacare', 'maconha': 'None', 'maconha_g': 25.0, 'cocaina': 'None', 'cocaina_g': 10.0, 'crack': 'None', 'crack_g': 0, 'ecstasy': 'None', 'ecstasy_g': 0, 'lsd': 'None', 'lsd_g': 0, 'outras': 'None', 'anabolizantes': <SimNao.NAO: 'Não'>, 'anorexigenos': <SimNao.NAO: 'Não'>, 'haxixe': <SimNao.NAO: 'Não'>, 'skank': <SimNao.NAO: 'Não'>, 'lanca_perfume': <SimNao.NAO: 'Não'>, 'tolueno': <SimNao.NAO: 'Não'>, 'den_drog': 'Sim', 'den_outros': 'Sim', 'sentenca': '5 anos em regime semiaberto', 'res_drogas': '25g de maconha, 10 comprimidos de ecstasy', 'res_outros': '', 'pena_base': '5 anos', 'agravantes33_agrup': <SimNao.SIM: 'Sim'>, 'confissao': <SimNao.SIM: 'Sim'>, 'menoridade': <SimNao.NAO: 'Não'>, 'atenuantes33_agrup': <SimNao.SIM: 'Sim'>, 'adolescente': <SimNao.NAO: 'Não'>, 'arma_de_fogo': <SimNao.NAO: 'Não'>, 'interestadual': <SimN

In [8]:
resultado_2 = parse_resposta(df_test["julgado"][0])
print(resultado_2)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_2881086/3739652588.py:14: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  format_instructions=SentencaModel.schema_json(indent=2)


{'processo': '00317367582710038016', 'juiz': 'Augusto Antonini', 'sexo_juiz': <SexoJuiz.MASCULINO: 'Masculino'>, 'vara': 'vara regional de justiça do trãsjecto', 'nome': 'Justiça Pública', 'local': 'São Paulo', 'maconha': 'Sim', 'maconha_g': None, 'cocaina': 'No', 'cocaina_g': 0.0, 'crack': 'Sim', 'crack_g': 193.0, 'ecstasy': 'Sim', 'ecstasy_g': 44.0, 'lsd': 'No', 'lsd_g': 0.0, 'outras': 'Jaque', 'anabolizantes': <SimNao.SIM: 'Sim'>, 'anorexigenos': <SimNao.SIM: 'Sim'>, 'haxixe': <SimNao.SIM: 'Sim'>, 'skank': <SimNao.SIM: 'Sim'>, 'lanca_perfume': <SimNao.SIM: 'Sim'>, 'tolueno': <SimNao.SIM: 'Sim'>, 'den_drog': 'Cocaina', 'den_outros': 'Sim', 'sentenca': 'Improcedente', 'res_drogas': 'Cocaina', 'res_outros': 'None', 'pena_base': '2 anos', 'agravantes33_agrup': <SimNao.SIM: 'Sim'>, 'confissao': <SimNao.SIM: 'Sim'>, 'menoridade': <SimNao.NAO: 'Não'>, 'atenuantes33_agrup': <SimNao.SIM: 'Sim'>, 'adolescente': <SimNao.NAO: 'Não'>, 'arma_de_fogo': <SimNao.NAO: 'Não'>, 'interestadual': <SimNao

In [2]:
from instructor import patch
from pydantic import BaseModel
from transformers import pipeline

# Define seu schema com Pydantic
class RelatorioVendas(BaseModel):
    mes: str
    ano: int
    total_vendas: float
    clientes: int

# Patch para compatibilidade com modelos locais (via Hugging Face)
patch(create=RelatorioVendas)

# Carrega o modelo
pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct",
    device_map="auto"
)

# Prompt estruturado
prompt = """
Documento: "Relatório de vendas: Agosto 2023. Total: R$ 75.200,50. Clientes atendidos: 142."
Extraia os dados no formato JSON seguindo este schema: {schema}
"""

# Executa a extração com validação Pydantic
response = pipe(
    prompt.format(schema=RelatorioVendas.schema_json()),
    max_new_tokens=512,
    return_full_text=False
)

# Valida e parseia via Pydantic
dados = RelatorioVendas.model_validate_json(response[0]["generated_text"])
print(dados.jsnon(indent=2))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
/tmp/ipykernel_2871299/1268210191.py:30: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  prompt.format(schema=RelatorioVendas.schema_json()),
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ValidationError: 1 validation error for RelatorioVendas
  Invalid JSON: trailing characters at line 12 column 1 [type=json_invalid, input_value='{\n  "properties": {\n  ... de 75.200,50,00, basta', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid

In [5]:
from outlines import models, generate
from pydantic import BaseModel

# Define o schema com Pydantic
class RelatorioVendas(BaseModel):
    mes: str
    ano: int
    total_vendas: float
    clientes: int

# Carrega o modelo Llama-3.2
model = models.transformers("meta-llama/Llama-3.2-3B-Instruct")

# Cria um gerador com restrição de JSON
generator = generate.json(model, RelatorioVendas)

# Executa a extração
documento = "Relatório de vendas: Março 2024. Total: R$ 120.000. Clientes: 89."
response = generator(f"Documento: {documento}\nSaída JSON:")

# Parseia o resultado
dados = RelatorioVendas.model_validate(response)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
import pandas as pddados.model_dump_json(indent=2)

SyntaxError: invalid syntax (2448261303.py, line 1)

In [15]:
from jsonformer import Jsonformer
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

schema = {
    "type": "object",
    "properties": {
        "mes": {"type": "string"},
        "ano": {"type": "number"},
        "total_vendas": {"type": "number"},
        "clientes": {"type": "number"}
    }
}

prompt = "Documento: Abril 2023. Total vendido: R$ 92.300. Clientes: 201. Saída JSON:"

jsonformer = Jsonformer(
    model,
    tokenizer,
    json_schema=schema,
    prompt=prompt
)

generated_data = jsonformer()

ImportError: cannot import name 'LogitsWarper' from 'transformers' (/home/228446@hertie-school.lan/workspace/.venv/lib/python3.10/site-packages/transformers/__init__.py)

In [12]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [47]:
# Configure o pipeline; defina max_new_tokens conforme o tamanho do seu contexto (ex.: 2048)
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=8048,
    #pad_token_id=tokenizer.pad_token_id,
    trust_remote_code=True
)

Device set to use cuda:0


In [16]:
from outlines import models, generate
from pydantic import BaseModel, ValidationError

class RelatorioVendas(BaseModel):
    mes: str
    ano: int
    total_vendas: float
    clientes: int

def extrair_dados(documento: str, max_retries=3):
    model = models.transformers("meta-llama/Llama-3.2-3B-Instruct")
    generator = generate.json(model, RelatorioVendas)
    
    for _ in range(max_retries):
        try:
            response = generator(f"Documento: {documento}\nSaída JSON:")
            return response.model_dump()
        except ValidationError as e:
            print(f"Tentativa falhou: {e}")
            continue
    raise RuntimeError("Falha após várias tentativas")

# Uso
dados = extrair_dados("Relatório de Março: R$ 95.000, 150 clientes")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
print(dados)

{'mes': 'MAR${ano}', 'ano': 2024, 'total_vendas': 95000.0, 'clientes': 150}


In [48]:
llm = HuggingFacePipeline(pipeline=llm_pipeline)

In [49]:
# Definindo o esquema de resposta
response_schemas = [
    ResponseSchema(name="processo", description="Número do processo, composto por 20 dígitos (exemplo: '00316684320178260050')"),
    ResponseSchema(name="juiz", description="Nome do juiz responsável"),
    ResponseSchema(name="sexo_juiz", description="Gênero do juiz (exemplo: 'Masculino' ou 'Feminino')"),
    ResponseSchema(name="vara", description="Vara criminal responsável (exemplo: 'Xª Vara Criminal')"),
    ResponseSchema(name="nome", description="Nome completo do réu"),
    ResponseSchema(name="local", description="Indicação do local relacionado ao réu ou à ocorrência"),
    ResponseSchema(name="maconha", description="Informação textual sobre a maconha apreendida; se não houver, utilize '0' ou 'None'"),
    ResponseSchema(name="maconha_g", description="Quantidade de maconha apreendida em gramas (numérico)"),
    ResponseSchema(name="cocaina", description="Informação textual sobre a cocaína apreendida; se não houver, utilize '0' ou 'None'"),
    ResponseSchema(name="cocaina_g", description="Quantidade de cocaína apreendida em gramas (numérico)"),
    ResponseSchema(name="crack", description="Informação textual sobre o crack apreendido; se não houver, utilize '0' ou 'None'"),
    ResponseSchema(name="crack_g", description="Quantidade de crack apreendida em gramas (numérico)"),
    ResponseSchema(name="ecstasy", description="Informação textual sobre o ecstasy apreendido"),
    ResponseSchema(name="ecstasy_g", description="Quantidade de ecstasy apreendida em gramas (numérico)"),
    ResponseSchema(name="lsd", description="Informação textual sobre o LSD apreendido"),
    ResponseSchema(name="lsd_g", description="Quantidade de LSD apreendida em gramas (numérico)"),
    ResponseSchema(name="outras", description="Outras drogas ou informações relacionadas; se não houver, utilize '0' ou 'None'"),
    ResponseSchema(name="anabolizantes", description="Indicação se houve apreensão de anabolizantes ('Sim' ou 'Não')"),
    ResponseSchema(name="anorexigenos", description="Indicação se houve apreensão de anorexígenos ('Sim' ou 'Não')"),
    ResponseSchema(name="haxixe", description="Indicação se houve apreensão de haxixe ('Sim' ou 'Não')"),
    ResponseSchema(name="skank", description="Indicação se houve apreensão de skank ('Sim' ou 'Não')"),
    ResponseSchema(name="lanca_perfume", description="Indicação se houve apreensão de lança-perfume ('Sim' ou 'Não')"),
    ResponseSchema(name="tolueno", description="Indicação se houve apreensão de tolueno ('Sim' ou 'Não')"),
    ResponseSchema(name="den_drog", description="Trecho referente à denúncia baseada em artigos da Lei de Drogas"),
    ResponseSchema(name="den_outros", description="Trecho referente à denúncia por outros artigos; se não houver, utilize 'None'"),
    ResponseSchema(name="sentenca", description="Resultado final da sentença"),
    ResponseSchema(name="res_drogas", description="Resultado relacionado aos aspectos de drogas"),
    ResponseSchema(name="res_outros", description="Resultado relacionado a outros aspectos; se não houver, utilize 'None'"),
    ResponseSchema(name="pena_base", description="Tempo definido como pena base"),
    ResponseSchema(name="agravantes33_agrup", description="Informação sobre agravantes ('Sim' ou 'Não')"),
    ResponseSchema(name="confissao", description="Indicação se houve confissão do réu ('Sim' ou 'Não')"),
    ResponseSchema(name="menoridade", description="Indicação se o réu é menor ('Sim' ou 'Não')"),
    ResponseSchema(name="atenuantes33_agrup", description="Informação sobre atenuantes ('Sim' ou 'Não')"),
    ResponseSchema(name="adolescente", description="Indicação se o réu é adolescente ('Sim' ou 'Não')"),
    ResponseSchema(name="arma_de_fogo", description="Indicação se houve apreensão de arma de fogo ('Sim' ou 'Não')"),
    ResponseSchema(name="interestadual", description="Indicação se o processo possui caráter interestadual ('Sim' ou 'Não')"),
    ResponseSchema(name="concurso_formal", description="Indicação se há concurso formal de crimes ('Sim' ou 'Não')"),
    ResponseSchema(name="estabelecimento", description="Indicação se houve apreensão em estabelecimento comercial ('Sim' ou 'Não')"),
    ResponseSchema(name="aumento33_agrup", description="Informação sobre aumento de pena ('Sim' ou 'Não')"),
    ResponseSchema(name="paragrafo_4o_agrupado", description="Informação agregada do parágrafo 4º"),
    ResponseSchema(name="pena33", description="Texto referente à pena"),
    ResponseSchema(name="pena33_meses", description="Pena convertida em meses (numérico)"),
    ResponseSchema(name="pena_drogas", description="Informação textual sobre pena relacionada a drogas; se não houver, utilize 'NA' ou 'None'"),
    ResponseSchema(name="pena_outros", description="Informação textual sobre pena relacionada a outros delitos; se não houver, utilize 'NA' ou 'None'"),
    ResponseSchema(name="tot_pen", description="Pena total em texto"),
    ResponseSchema(name="tot_pen_meses", description="Pena total convertida em meses (numérico)"),
    ResponseSchema(name="substituicao_da_pena", description="Informação sobre eventual substituição da pena; se não houver, utilize 'NA' ou 'None'"),
    ResponseSchema(name="regime_inicial", description="Regime inicial da pena (valores possíveis: 'Fechado', 'Semi-aberto', 'Aberto' ou 'None')"),
    ResponseSchema(name="flag_local_de_trafico", description="True se o local relacionado à ocorrência está associado ao tráfico"),
    ResponseSchema(name="flag_preso_no_momento_da_sentenca", description="True se o réu estava preso no momento da sentença"),
    ResponseSchema(name="flag_confissao_informal", description="True se o réu apresentou uma confissão informal"),
    ResponseSchema(name="flag_confissao", description="True se houver evidências de uma confissão formal"),
    ResponseSchema(name="flag_denuncia_anonima", description="True se a denúncia foi realizada de forma anônima"),
    ResponseSchema(name="flag_denuncia", description="True se houver menção a uma denúncia formal"),
    ResponseSchema(name="flag_atitude_suspeita", description="True se o texto descrever comportamentos suspeitos"),
    ResponseSchema(name="flag_divergencias_nos_relatos_dos_policiais", description="True se houver divergências nos relatos dos policiais"),
    ResponseSchema(name="flag_investigacao", description="True se houver menção a investigação"),
    ResponseSchema(name="flag_interceptacao", description="True se o texto mencionar interceptações"),
    ResponseSchema(name="flag_mandado", description="True se houver referência à expedição ou execução de mandado"),
    ResponseSchema(name="flag_nacionalidade", description="True se a nacionalidade do réu for relevante"),
    ResponseSchema(name="flag_revista_vexatoria", description="True se houver menção a revista vexatória"),
    ResponseSchema(name="aval_antecedentes", description="True se houver referência a antecedentes criminais"),
    ResponseSchema(name="aval_conduta", description="True se houver avaliação sobre a conduta do réu"),
    ResponseSchema(name="aval_personalidade", description="True se houver menção à personalidade do réu"),
    ResponseSchema(name="aval_natureza", description="True se o texto avaliar a natureza do delito"),
    ResponseSchema(name="aval_quantidade", description="True se houver avaliação da quantidade de drogas"),
    ResponseSchema(name="aval_variedade", description="True se houver avaliação sobre a diversidade de substâncias"),
    ResponseSchema(name="aval_circunstancias", description="True se o trecho de avaliação abordar as circunstâncias do caso"),
    ResponseSchema(name="aval_consequencias", description="True se houver avaliação das consequências do delito"),
    ResponseSchema(name="aval_culpabilidade", description="True se houver avaliação da culpabilidade do réu")
]


In [50]:
response_schemas

[ResponseSchema(name='processo', description="Número do processo, composto por 20 dígitos (exemplo: '00316684320178260050')", type='string'),
 ResponseSchema(name='juiz', description='Nome do juiz responsável', type='string'),
 ResponseSchema(name='sexo_juiz', description="Gênero do juiz (exemplo: 'Masculino' ou 'Feminino')", type='string'),
 ResponseSchema(name='vara', description="Vara criminal responsável (exemplo: 'Xª Vara Criminal')", type='string'),
 ResponseSchema(name='nome', description='Nome completo do réu', type='string'),
 ResponseSchema(name='local', description='Indicação do local relacionado ao réu ou à ocorrência', type='string'),
 ResponseSchema(name='maconha', description="Informação textual sobre a maconha apreendida; se não houver, utilize '0' ou 'None'", type='string'),
 ResponseSchema(name='maconha_g', description='Quantidade de maconha apreendida em gramas (numérico)', type='string'),
 ResponseSchema(name='cocaina', description="Informação textual sobre a cocaín

In [51]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
format_instructions = format_instructions.replace("{", "{{").replace("}", "}}")


In [52]:
output_parser
format_instructions

'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{{\n\t"processo": string  // Número do processo, composto por 20 dígitos (exemplo: \'00316684320178260050\')\n\t"juiz": string  // Nome do juiz responsável\n\t"sexo_juiz": string  // Gênero do juiz (exemplo: \'Masculino\' ou \'Feminino\')\n\t"vara": string  // Vara criminal responsável (exemplo: \'Xª Vara Criminal\')\n\t"nome": string  // Nome completo do réu\n\t"local": string  // Indicação do local relacionado ao réu ou à ocorrência\n\t"maconha": string  // Informação textual sobre a maconha apreendida; se não houver, utilize \'0\' ou \'None\'\n\t"maconha_g": string  // Quantidade de maconha apreendida em gramas (numérico)\n\t"cocaina": string  // Informação textual sobre a cocaína apreendida; se não houver, utilize \'0\' ou \'None\'\n\t"cocaina_g": string  // Quantidade de cocaína apreendida em gramas (numérico)\n\t"crack": string  // I

In [53]:
# 3. Ler o prompt detalhado de um arquivo .txt (instruções para extração)
with open("prompt/prompt_v2.txt", "r", encoding="utf-8") as f:
    prompt_text = f.read()


In [54]:
# prompt_template = f"""
# {prompt_text}
# Extraia os dados do texto fornecido e retorne a resposta em formato JSON válido.
# {format_instructions.replace("{", "{{").replace("}", "}}")}

# Texto da sentença:
# {{text}}
# """

# Constrói o template via concatenação de strings (apenas {text} é o placeholder)
prompt_template = (
    "Você é um assistente jurídico especialista em extrair informações de sentenças judiciais.\n"
    "Extraia os dados do texto fornecido e retorne a resposta em formato JSON válido.\n"
    "Siga as instruções abaixo e garanta que a saída seja um JSON válido conforme o seguinte esquema:\n\n"
    + format_instructions +
    "\n\nImportante: Sua resposta **deve** ser um bloco de código markdown iniciado com ```json e terminado com ```.\n\n"
    "Texto da sentença:\n{text}"
)

#print(prompt_template)  # Verifique o prompt final


prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["text"]
)

#print(prompt_template)

In [55]:
# 2. Ler o arquivo .parquet com as decisões judiciais
df = pd.read_parquet("validation.parquet")
df_test = df[0:2].copy(deep=True).reset_index(drop=True)

In [56]:
chain = LLMChain(llm=llm, prompt=prompt, output_parser=output_parser)

In [57]:
sentenca = df_test["julgado"][0]

In [58]:
# chain.invoke({"text": sentenca})

try:
    result = chain.invoke({"text": sentenca})
    print("Resultado parseado:", result)
except Exception as e:
    print("Erro durante a invocação:", e)
    # Aqui você pode salvar o raw_output ou tomar outra ação para debugar



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Erro durante a invocação: Got invalid return object. Expected key `processo` to be present, but got  and 
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 


In [ ]:
results = []
for idx, row in df_test.iterrows():
    decision_text = row["julgado"]
    print(f"Processing decision {idx} (words: {len(decision_text.split())})")
    
    # Construa a entrada no formato de conversa:
    # Por exemplo, usando as tags "System:" e "User:" e adicionando "Assistant:" para indicar onde a resposta deve iniciar.
    conversation = (
        {"role": "system", "content": {prompt_text}},
        {"role": "user", "content": {decision_text}}
    )
    
    output = llm_pipeline(conversation)
    generated_text = output[0]["generated_text"][2]['content'] # output[0]["generated_text"]
    results.append(generated_text)
    print(f"Decision {idx} processed:\n{generated_text}\n{'-'*60}\n")



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processing decision 0 (words: 625)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Decision 0 processed:
```json
[
  {
    "processo": "0031736-90.2017.8.26.0050",
    "juiz": "Augusto Antonini",
    "sexo_juiz": "Masculino",
    "vara": "Xª Vara Criminal",
    "nome": "GUSTAVO RODRIGUES PATRÍCIO",
    "local": "Rua João da Cunha Lobo, altura do nº 42, Cangaíba",
    "maconha": "182 pinos de cocaína e 24 invólucros de maconha",
    "maconha_g": "None",
    "cocaina": "182 pinos de cocaína",
    "cocaina_g": "None",
    "crack": "None",
    "crack_g": "None",
    "ecstasy": "None",
    "ecstasy_g": "None",
    "lsd": "None",
    "lsd_g": "None",
    "outras": "None",
    "anabolizantes": "None",
    "anorexigenos": "None",
    "haxixe": "None",
    "skank": "None",
    "lanca_perfume": "None",
    "tolueno": "None",
    "den_drog": "Cocaína e Maconha",
    "den_outros": "None",
    "sentenca": "ABSOLVER",
    "res_drogas": "Materialidade do delito comprovada pelo Exame Químico Toxicológico",
    "res_outros": "Prova da autoria insuficiente",
    "pena_base": "None",
 